In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "bank_dev")
dbutils.widgets.text("esquema_bronze", "bronze")
dbutils.widgets.text("esquema_silver", "silver")
dbutils.widgets.text("storageName", "saprojectbankdeveastus")
dbutils.widgets.text("containerSilver", "silver")

In [0]:
catalogo         = dbutils.widgets.get("catalogo")
esquema_bronze   = dbutils.widgets.get("esquema_bronze")
esquema_silver   = dbutils.widgets.get("esquema_silver")
storageName      = dbutils.widgets.get("storageName")
containerSilver  = dbutils.widgets.get("containerSilver")

In [0]:
checkpoint_location_b = f"abfss://bronze@{storageName}.dfs.core.windows.net/_checkpoints/transactions"
checkpoint_location_s = f"abfss://{containerSilver}@{storageName}.dfs.core.windows.net/_checkpoints/transactions" 

tabla_bronze = f"{catalogo}.{esquema_bronze}.transactions"
tabla_silver = f"{catalogo}.{esquema_silver}.transactions"

In [0]:
df = spark.read.table("bank_dev.bronze.transactions")



In [0]:
df_bronze = spark.readStream.table(f"{catalogo}.{esquema_bronze}.transactions")


In [0]:
# Transformacion y casteo... :
df_silver_tranform = df_bronze \
    .withColumn("id", F.col("id").cast("bigint")) \
    .withColumn("client_id", F.col("client_id").cast("bigint")) \
    .withColumn("card_id", F.col("card_id").cast("bigint")) \
    .withColumn("merchant_id", F.col("merchant_id").cast("bigint")) \
    .withColumn("mcc", F.coalesce(F.col("mcc").cast("int"), F.lit(9999))) \
    .withColumn("date", F.to_timestamp(F.col("date"))) \
    .withColumn("amount", F.regexp_replace(F.col("amount"), r"[^0-9.-]", "").cast("decimal(18,2)")) \
    .withColumn("merchant_state", F.coalesce(F.col("merchant_state"), F.lit("N/A"))) \
    .withColumn("zip", F.coalesce(F.col("zip").cast("string"), F.lit("N/A"))) \
    .withColumn("tiene_error", F.col("errors").isNotNull()) \
    .withColumn("lista_errores", 
                F.when(F.col("errors").isNotNull(), F.split(F.col("errors"), ","))
                 .otherwise(F.array().cast("array<string>"))) \
    .withColumn("silver_load_date", F.current_timestamp()) \
    .drop("_rescued_data", "__is_error", "errors")

In [0]:
(df_silver_tranform.writeStream
 .format("delta")
 .outputMode("append")
 .option("checkpointLocation", checkpoint_location_s)
 .trigger(availableNow=True) # <--- ESTO ES LA CLAVE
 .toTable(tabla_silver)) # 